# Reference Resolution Dev Notebook

First attempt at integrating methods directly into METER's pytorch lightning framework. 

In [86]:
import random
import io
import pyarrow as pa
import os
import copy
import pytorch_lightning as pl
from sacred import Experiment
from PIL import Image
from tqdm import tqdm
import numpy as np
import skimage.io as skio
import matplotlib.pyplot as plt
from refer import REFER
import pandas as pd

import torch
from torch.optim import AdamW
from torch.utils.data import DataLoader
from pytorch_lightning import LightningDataModule

from transformers import ElectraTokenizer

from refcoco_utils import get_bounded_subimage
from refcoco_utils import _config
from refcoco_utils import _loss_names

from meter.transforms import keys_to_transforms
from meter.config import ex
from meter.modules import METERTransformerSS
from meter.datamodules.multitask_datamodule import MTDataModule
from meter.datasets.base_dataset import BaseDataset

## RefCOCO Data

### Utility Functions

In [4]:
def get_sent_ids(refer):
    sent_ids = []
    for ref_id in refer.getRefIds():
        ref = refer.Refs[ref_id]
        for sent_id in ref['sent_ids']:
            sent_ids.append(sent_id)
    return sent_ids

### Import  Data

In [5]:
data_root = '/home/claytonfields/nlp/code/data/coco'  # contains refclef, refcoco, refcoco+, refcocog and images
dataset = 'refcoco' 
splitBy = 'unc'
refer = REFER(data_root, dataset, splitBy)

loading dataset refcoco into memory...
testing
creating index...
index created.
DONE (t=9.97s)


In [6]:
refer.IMAGE_DIR = '/home/claytonfields/nlp/code/data/coco/images/mscoco/train2014'

In [7]:
config = copy.deepcopy(_config)

### Find Max Number of Objects

In [8]:
train_ids = refer.getRefIds()
size = []
for ref_id in train_ids:
    ref = refer.Refs[ref_id]
    img_id = ref['image_id']
    ann_id = ref['ann_id']
    objs = refer.imgToAnns[img_id]
    size.append(len(objs))
num_refs = len(train_ids)
max_size = np.max(size)
avg_size = np.mean(size)
median_size = np.median(size)
p_70 =  np.percentile(size, 70)
p_80 =  np.percentile(size, 80)
p_90 =  np.percentile(size, 90)
p_95 =  np.percentile(size, 95)
p_99 =  np.percentile(size, 99)
num_over_42 = np.sum(np.array(size) >= 42)

print(f'The most objects in any reference is {max_size}')
print(f'The average number of objects in each reference is {avg_size}')
print(f'The median number of objects in each reference is {median_size}')
print()
print(f'The 70th percentile of the number of objects in each reference is {p_70}')
print(f'The 80th percentile of the number of objects in each reference is {p_80}')
print(f'The 80th percentile of the number of objects in each reference is {p_90}')
print(f'The 95th percentile of the number of objects in each reference is {p_95}')
print(f'The 99th percentile of the number of objects in each reference is {p_99}')
print()
print(f'A max_bb of 42 would exclude {num_over_42} of {num_refs} refs')

The most objects in any reference is 75
The average number of objects in each reference is 10.60916
The median number of objects in each reference is 8.0

The 70th percentile of the number of objects in each reference is 13.0
The 80th percentile of the number of objects in each reference is 16.0
The 80th percentile of the number of objects in each reference is 21.0
The 95th percentile of the number of objects in each reference is 26.0
The 99th percentile of the number of objects in each reference is 41.0

A max_bb of 42 would exclude 453 of 50000 refs


### Find Max Number of Sentences

In [9]:
size = []
for ref_id in train_ids:
    ref = refer.Refs[ref_id]
    objs = refer.imgToAnns[img_id]
    size.append(len(ref['sentences']))
max_size = max(size)
print(f'The most sentences in any reference is {max_size}')

The most sentences in any reference is 6


In [71]:
_config = {  
    "exp_name":"finetune_ref",
    "seed" : 0,
    # "datasets" : ["coco", "vg", "sbu", "gcc"],
    # "datasets" : ["coco", "vg"],
    "datasets" : ["coco"],
    "loss_names" :{'itm': 0,
    'mlm': 0,
    'mpp': 0,
    'vqa': 0,
    'vcr': 0,
    'vcr_qar': 0,
    'nlvr2': 0,
    'irtr': 0,
    'contras': 0,
    'snli': 0,
    'ref': 1},
    "batch_size" : 10,  # this is a desired batch size; pl trainer will accumulate gradients when per step batch is smaller.

    # Image setting
    "train_transform_keys" : ["imagenet"],
    "val_transform_keys" : ["imagenet"],
    "image_size" : 224,
    "patch_size" : 16,
    "draw_false_image" : 1,
    "image_only" : False,
    "resolution_before" : 224,

    # Text Setting
    "vqav2_label_size" : 3129,
    "max_text_len" : 40,
    "tokenizer" : "google/electra-small-discriminator",
    "vocab_size" : 30522,
    "whole_word_masking" : False, # note that whole_word_masking does not work for RoBERTa
    "mlm_prob" : 0.15,
    "draw_false_text" : 0,

    # Transformer Setting
    "num_top_layer" : 6,
    "input_image_embed_size" : 192,
    "input_text_embed_size" : 256,
    "vit" : "vit_deit_tiny_patch16_224",
    "hidden_size" : 256,
    "num_heads" : 4,
    "num_layers" : 6,
    "mlp_ratio" : 4,
    "drop_rate" : 0.1,

    # Optimizer Setting
    "optim_type" : "adamw",
    "learning_rate" : 1e-5,
    "weight_decay" : 0.01,
    "decay_power" : 1,
    "max_epoch" : 3,
    "max_steps" : 100000,
    "warmup_steps" : 10000,
    "end_lr" : 0,
    "lr_mult_head" : 5,  # multiply lr for downstream heads
    "lr_mult_cross_modal" : 5,  # multiply lr for the cross-modal module

    # Downstream Setting
    "get_recall_metric" : False,
    
    
    "model_type" : "METER",

    # PL Trainer Setting
    "resume_from" : None,
    "fast_dev_run" : False,
    "val_check_interval" : 1.0,
    "test_only" : False,

    "data_root" : data_root,
    "log_dir" : "result",
    "per_gpu_batchsize" : 3,  # you should define this manually with per_gpu_batch_size:#
    "num_gpus" : num_gpus,
    "num_nodes" : 1,
    "load_path" : load_path,
    "num_workers" : 12,
    "precision" : 16
}

## Data Class for Ref Res with multiple samples

1. Adjust ref_res_classifier dimension to reflect max_bb

2.  May require padding to some max_num_bb.

The max number of objects in any ref in the training set is 75. Should consider some smaller number like 42. 42 represents the 99th percentile
    
**Below is a previous dataset class for reference**

**New class that produces uniform items padded to max_bb**

In [280]:
class RefcocoDataset(torch.utils.data.Dataset):

    def __init__(self, refer, tokenizer, device='cpu', errors=None, split='', max_bb = 42):
        self.tokenizer = tokenizer
        self.refer = refer
        self.max_bb = max_bb
        self.device = device
        self.errors = errors
        self.split = split
        self.sent_ids = self.get_sent_ids()
        self.duds = []

    def __len__(self):
        return len(self.sent_ids)
    
    def get_sent_ids(self):
        sent_ids = []
        for ref_id in self.refer.getRefIds(split=self.split):
            
            ref = self.refer.Refs[ref_id]
            img_id = ref['image_id']
            objs = refer.imgToAnns[img_id]
            if len(objs) <= self.max_bb:
                for sent_id in ref['sent_ids']:
                    if not sent_id in self.errors:
                        sent_ids.append(sent_id)
        return sent_ids
    
    def __getitem__(self, index):
        max_bb = self.max_bb
        
        sent_id = self.sent_ids[index]
        ref = self.refer.sentToRef[sent_id]
        sent = self.refer.Sents[sent_id]
        
        img_id = ref['image_id']
        ann_id = ref['ann_id']
        objs = refer.imgToAnns[img_id]
        obj_ids = [obj['id'] for obj in objs]
#         obj_pad = [0 for _ in range(max_bb-len(obj_ids))]
#         obj_ids_total = obj_ids+obj_pad

        sub_images = []
        for obj in objs:
            x_a = get_bounded_subimage(refer, img_id, obj['id'], xs=224,ys=224, show=False)
            if x_a is not None:
                sub_images.append(x_a)
        
        num_sub_images = len(sub_images)
        num_pad = max_bb - num_sub_images 
        
#         pad_image = torch.zeros(1,3,224,224)
#         for _ in range(max_bb - num_sub_images):
#             sub_images.append(pad_image)
        
        # text ids
        ids = self.tokenizer.encode(
            sent['sent'],
            padding="max_length",
            truncation=True,
            max_length=40,
            return_special_tokens_mask=True,
        )
        ids = ids
        text_ids = torch.tensor(ids).repeat(num_sub_images,1)
#         pad_ids =  torch.zeros(num_pad,40)
#         text_ids = torch.cat((repeat_ids, pad_ids)).to(torch.long)
        # text masks
        num_tokens = torch.where(text_ids[0] > 0)[0].size(dim=0)
        masks = torch.cat((torch.ones(num_tokens), torch.zeros(40-num_tokens))).to(torch.long)
        text_masks = masks.repeat(num_sub_images,1)
#         pad_masks = torch.zeros(num_pad, 40)
#         text_masks = torch.cat((repeat_masks, pad_masks)).to(torch.long)
        # text_labels
        labels = torch.full((40,),-100)
        text_labels = labels.repeat(num_sub_images, 1)
#         pad_labels = torch.zeros(num_pad, 40)
#         text_labels = torch.cat((repeat_labels, pad_labels)).to(torch.long)
        
        target = torch.tensor([obj_ids.index(ann_id)])

        return_dict = {
            'ann_id' : ann_id,
            'image' : sub_images,#.to(self.device)],
            'obj_ids' : torch.tensor(obj_ids),#.to(self.device),
            'target' : target,#.to(self.device),
            'num_bb' : num_sub_images,
            'text' : sent['sent'],
            'text_ids' : text_ids,#.to(self.device),
            'text_labels' : text_labels,#.to(self.device),
            'text_masks' : text_masks,#.to(self.device)
        }
        
        return return_dict




In [338]:
def collate(batch):
    targets = []
    for b in batch:
        targets.append(b['target'])
    targets = torch.tensor(targets)
    
    max_bb = max([b['num_bb'] for b in batch])

    pad_image = torch.zeros(1,3,224,224)
    
    for b in batch:
        num_examples = b['num_bb']
        num_pad = max_bb - num_examples


        for _ in range(num_pad):
            b['image'].append(pad_image)
        b['image'] = [torch.cat(b['image'])]

        pad_masks = torch.zeros(num_pad, 40)
        b['text_masks'] = torch.cat((b['text_masks'], pad_masks)).to(torch.long)


        pad_labels = torch.zeros(num_pad, 40)
        b['text_labels'] = torch.cat((b['text_labels'], pad_labels)).to(torch.long)

        pad_ids =  torch.zeros(num_pad,40)
        b['text_ids'] = torch.cat((b['text_ids'], pad_ids)).to(torch.long)
    
        return (batch, targets)
    

In [339]:
errors_df = pd.read_csv('Errors.csv')
errors_list = errors_df['Sent ID'].to_list()
# dm = RefcocoDataModule(config, refer, device, errors_list, collate)
ds = RefcocoDataset(refer, tokenizer, device, errors_list, split='train', max_bb=42)
train_params = {'batch_size': BATCH_SIZE,
                'shuffle': False,
                'num_workers': 0,
                'collate_fn' : collate
                }
training_loader = torch.utils.data.DataLoader(ds, **train_params)

In [340]:
batch = next(iter(training_loader))

In [341]:
batch[-1]

tensor([16, 16, 16,  0,  0,  0,  2,  2,  2,  1])

In [345]:
batch[0]

[{'ann_id': 1719310,
  'image': [tensor([[[[0.0549, 0.0549, 0.0549,  ..., 0.0078, 0.0000, 0.0000],
             [0.0667, 0.0667, 0.0627,  ..., 0.0118, 0.0118, 0.0118],
             [0.0627, 0.0627, 0.0627,  ..., 0.0039, 0.0039, 0.0039],
             ...,
             [0.2824, 0.2588, 0.2157,  ..., 0.6784, 0.7176, 0.7412],
             [0.2745, 0.2549, 0.2235,  ..., 0.8588, 0.8627, 0.8627],
             [0.2745, 0.2588, 0.2353,  ..., 0.6941, 0.6745, 0.6627]],
   
            [[0.0510, 0.0510, 0.0510,  ..., 0.0118, 0.0078, 0.0039],
             [0.0667, 0.0667, 0.0627,  ..., 0.0118, 0.0118, 0.0118],
             [0.0627, 0.0627, 0.0627,  ..., 0.0039, 0.0000, 0.0000],
             ...,
             [0.4314, 0.3922, 0.3216,  ..., 0.7804, 0.8196, 0.8471],
             [0.4157, 0.3843, 0.3255,  ..., 0.8863, 0.8706, 0.8627],
             [0.4157, 0.3882, 0.3373,  ..., 0.6471, 0.6078, 0.5843]],
   
            [[0.0314, 0.0314, 0.0314,  ..., 0.0039, 0.0039, 0.0039],
             [0.0353, 0.035

In [346]:
targets = batch[1]
batch = batch[0]

In [347]:
len(batch)

10

In [348]:
batch[5]['image'].__len__()

33

In [351]:
batch[0]['image'].__len__()

1

In [315]:
max_bb = max([b['num_bb'] for b in batch])

pad_image = torch.zeros(1,3,224,224)
ret = []
for b in batch:
    num_examples = b['num_bb']
    num_pad = max_bb - num_examples


    for _ in range(num_pad):
        b['image'].append(pad_image)
    b['image'] = [torch.cat(b['image'])]

    pad_masks = torch.zeros(num_pad, 40)
    b['text_masks'] = torch.cat((b['text_masks'], pad_masks)).to(torch.long)


    pad_labels = torch.zeros(num_pad, 40)
    b['text_labels'] = torch.cat((b['text_labels'], pad_labels)).to(torch.long)

    pad_ids =  torch.zeros(num_pad,40)
    b['text_ids'] = torch.cat((b['text_ids'], pad_ids)).to(torch.long)

    

In [317]:
batch[-1]

{'ann_id': 485695,
 'image': [tensor([[[[0.7647, 0.7451, 0.7098,  ..., 0.9529, 0.9490, 0.9412],
            [0.7490, 0.7412, 0.7255,  ..., 0.9765, 0.9569, 0.9412],
            [0.7137, 0.7216, 0.7333,  ..., 0.9765, 0.9529, 0.9412],
            ...,
            [0.8157, 0.7961, 0.7686,  ..., 0.8353, 0.8118, 0.8000],
            [0.7804, 0.7725, 0.7686,  ..., 0.7686, 0.7804, 0.7922],
            [0.7098, 0.7294, 0.7647,  ..., 0.7843, 0.8000, 0.8157]],
  
           [[0.6941, 0.6706, 0.6235,  ..., 0.8941, 0.8863, 0.8784],
            [0.6745, 0.6667, 0.6392,  ..., 0.9216, 0.8980, 0.8824],
            [0.6392, 0.6471, 0.6510,  ..., 0.9255, 0.9020, 0.8863],
            ...,
            [0.7529, 0.7333, 0.7059,  ..., 0.7647, 0.7373, 0.7294],
            [0.7176, 0.7098, 0.7059,  ..., 0.6980, 0.7020, 0.7176],
            [0.6471, 0.6667, 0.7020,  ..., 0.7216, 0.7333, 0.7451]],
  
           [[0.4588, 0.4392, 0.4078,  ..., 0.8275, 0.7922, 0.7725],
            [0.4431, 0.4353, 0.4196,  ..., 0.8

**Data Module for pytorch lightning**

## METER Model

In [73]:
config = copy.deepcopy(_config)
pl.seed_everything(_config["seed"])
dm = RefcocoDataModule(config, refer, collate_fn)
model = METERTransformerSS(_config)
# model.current_tasks.append('ref')

Global seed set to 0
Some weights of the model checkpoint at google/electra-small-discriminator were not used when initializing ElectraModel: ['discriminator_predictions.dense_prediction.bias', 'discriminator_predictions.dense.weight', 'discriminator_predictions.dense_prediction.weight', 'discriminator_predictions.dense.bias']
- This IS expected if you are initializing ElectraModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing ElectraModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


## Perform Ref Res with METER

**Define key variables and parameters**

In [74]:
optim = AdamW(model.parameters(), lr=1e-4)
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

# Ref Res with METER
tokenizer = ElectraTokenizer.from_pretrained('google/electra-small-discriminator')
BATCH_SIZE = 10


epochs = 1
# loader = dm.train_dataloader()
optim = AdamW(model.parameters(), lr=1e-4)
loss_fn = torch.nn.functional.cross_entropy
# device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
dvice  = torch.device('cpu')

In [148]:
ds = RefcocoDataset(refer, tokenizer, split='train', max_bb=42)
# ds = NewRefcocoDataset(refer, tokenizer)
train_params = {'batch_size': BATCH_SIZE,
                'shuffle': False,
                'num_workers': 0,
                'collate_fn' : collate
                }

training_loader = torch.utils.data.DataLoader(ds, **train_params)

In [76]:
model.current_tasks = ['ref']

In [80]:
logits

tensor([[0.0756],
        [0.3122],
        [0.3353],
        [0.4429],
        [0.4583],
        [0.4298],
        [0.4356],
        [0.4314],
        [0.4636],
        [0.4221],
        [0.4607],
        [0.4419],
        [0.4460],
        [0.4462],
        [0.4172],
        [0.4439],
        [0.1519],
        [0.4601],
        [0.4164],
        [0.4328],
        [0.4183],
        [0.4411],
        [0.4261],
        [0.4311],
        [0.4409],
        [0.4459],
        [0.4236],
        [0.4545],
        [0.4697],
        [0.3565],
        [0.4325],
        [0.4026],
        [0.3064],
        [0.4131],
        [0.4054],
        [0.4386],
        [0.3467],
        [0.3852],
        [0.4195],
        [0.4579],
        [0.3366],
        [0.3920]], grad_fn=<AddmmBackward0>)

### Training

**Training Loop Dev Cell**

In [ ]:
pl.seed_everything(_config["seed"])

# dm = MTDataModule(_config, dist=False)

model = METERTransformerSS(_config)
exp_name = f'{_config["exp_name"]}'

os.makedirs(_config["log_dir"], exist_ok=True)
checkpoint_callback = pl.callbacks.ModelCheckpoint(
    save_top_k=1,
    verbose=True,
    monitor="val/the_metric",
    mode="max",
    save_last=True,
)
logger = pl.loggers.TensorBoardLogger(
    _config["log_dir"],
    name=f'{exp_name}_seed{_config["seed"]}_from_{_config["load_path"].split("/")[-1][:-5]}',
)

lr_callback = pl.callbacks.LearningRateMonitor(logging_interval="step")
callbacks = [checkpoint_callback, lr_callback]

num_gpus = (
    _config["num_gpus"]
    if isinstance(_config["num_gpus"], int)
    else len(_config["num_gpus"])
)

grad_steps = max(_config["batch_size"] // (
    _config["per_gpu_batchsize"] * num_gpus * _config["num_nodes"]
), 1)

max_steps = _config["max_steps"] if _config["max_steps"] is not None else None

trainer = pl.Trainer(
    gpus=0,
    num_nodes=_config["num_nodes"],
    precision=_config["precision"],
#     accelerator="cpu",
    benchmark=True,
    deterministic=True,
    max_epochs=_config["max_epoch"] if max_steps is None else 1000,
    max_steps=max_steps,
    callbacks=callbacks,
    logger=logger,
    #prepare_data_per_node=False,
    #replace_sampler_ddp=False,
    accumulate_grad_batches=grad_steps,
    log_every_n_steps=10,
    flush_logs_every_n_steps=10,
#     resume_from_checkpoint=_config["resume_from"],
    weights_summary="top",
    fast_dev_run=_config["fast_dev_run"],
    val_check_interval=_config["val_check_interval"],
)

if not _config["test_only"]:
    trainer.fit(model, datamodule=dm)
else:
    trainer.test(model, datamodule=dm)

Global seed set to 0
GPU available: True, used: False
TPU available: False, using: 0 TPU cores
/home/claytonfields/anaconda3/envs/vl/lib/python3.7/site-packages/pytorch_lightning/utilities/distributed.py:69: UserWarning: GPU available but not used. Set the gpus flag in your trainer `Trainer(gpus=1)` or script `--gpus=1`.
  warnings.warn(*args, **kwargs)

   | Name                        | Type              | Params
-------------------------------------------------------------------
0  | cross_modal_text_transform  | Linear            | 65.8 K
1  | cross_modal_image_transform | Linear            | 49.4 K
2  | token_type_embeddings       | Embedding         | 512   
3  | vit_model                   | VisionTransformer | 5.5 M 
4  | text_transformer            | ElectraModel      | 13.5 M
5  | cross_modal_image_layers    | ModuleList        | 6.3 M 
6  | cross_modal_text_layers     | ModuleList        | 6.3 M 
7  | cross_modal_image_pooler    | Pooler            | 65.8 K
8  | cross_modal_

Validation sanity check: 0it [00:00, ?it/s]

In [ ]:
batch = next(iter(training_loader))

In [ ]:
model(batch)

**TODO:** Work on training that is compatible with METER's pytorch lightning cofiguration

In [ ]:
model.train()
# losses = []
# logit_list = []
# targets = []
optim.zero_grad()
for batch in training_loader:
    try:
        model(batch)
        
    except RuntimeError:
        print(f'RuntimeError')
loss = loss_fn(logits.reshape(1,-1),target)
losses.append(loss.item())
loss.backward()
optim.step()

## Test Cells:

In [27]:
model.current_tasks = vqa 

[]

In [23]:
model

METERTransformerSS(
  (cross_modal_text_transform): Linear(in_features=256, out_features=256, bias=True)
  (cross_modal_image_transform): Linear(in_features=192, out_features=256, bias=True)
  (token_type_embeddings): Embedding(2, 256)
  (vit_model): VisionTransformer(
    (patch_embed): PatchEmbed(
      (proj): Conv2d(3, 192, kernel_size=(16, 16), stride=(16, 16))
      (norm): Identity()
    )
    (pos_drop): Dropout(p=0.0, inplace=False)
    (blocks): ModuleList(
      (0): Block(
        (norm1): LayerNorm((192,), eps=1e-06, elementwise_affine=True)
        (attn): Attention(
          (qkv): Linear(in_features=192, out_features=576, bias=True)
          (attn_drop): Dropout(p=0.0, inplace=False)
          (proj): Linear(in_features=192, out_features=192, bias=True)
          (proj_drop): Dropout(p=0.0, inplace=False)
        )
        (drop_path): Identity()
        (norm2): LayerNorm((192,), eps=1e-06, elementwise_affine=True)
        (mlp): Mlp(
          (fc1): Linear(in_featu